# T3 · Storage Arithmetic
Subtasks: Part A (three ways, five row counts), Part B (the finding), Part C (in context). All in this one notebook.

**Model:** 2 bytes per character, 4 bytes per integer key. Only province and region information is counted.

In [1]:
import pandas as pd, os
from fractions import Fraction as Fr
pd.set_option("display.max_colwidth", None, "display.width", 250)

d = next(x for x in [".", "..", "/mnt/user-data/uploads"] if os.path.exists(os.path.join(x, "DimCity.csv")))
flat = pd.read_csv(os.path.join(d, "DimCity.csv"))
DimProvince = pd.read_csv(os.path.join(d, "DimProvince.csv"))
Fact = pd.read_csv(os.path.join(d, "FactShipment.csv"))
assert Fact.WeightKG.sum() == 578_650 and Fact.FreightCostPKR.sum() == 4_808_000     # source untouched

CHAR, KEY = 2, 4
real = flat[flat.CityKey != -1]
n_prov, n_reg = real.Province.nunique(), real.Region.nunique()
print("Distinct provinces:", n_prov, "| distinct regions:", n_reg, "(excluding Unknown)")

# --- inputs counted from the data ---
dict_chars   = sum(map(len, real.Province.unique())) + sum(map(len, real.Region.unique()))      # each distinct string once
raw_real_chars = (real.Province.str.len() + real.Region.str.len()).sum()                          # text on the 14 real city rows
raw_unknown_chars = len("Unknown") * 2                                                          # text on the Unknown city row
dimprov_bytes = (len(DimProvince) * 2 * KEY                                                     # ProvinceKey + CountryKey on every row
                 + CHAR * DimProvince.ProvinceName.str.len().sum()                              # names
                 + CHAR * DimProvince.RegionName.str.len().sum())                               # regions (written on each row)
print("Dictionary characters:", dict_chars, "| DimProvince rows (with Unknown):", len(DimProvince), "| DimProvince bytes:", dimprov_bytes)

Distinct provinces: 5 | distinct regions: 4 (excluding Unknown)
Dictionary characters: 74 | DimProvince rows (with Unknown): 6 | DimProvince bytes: 232


## Part A · Three ways to hold it

In [2]:
def flat_raw(n):     return CHAR * (Fr(int(raw_real_chars), len(real)) * (n - 1) + raw_unknown_chars)   # average real row, plus the Unknown row
def flat_encoded(n): return CHAR * dict_chars + 2 * KEY * n          # dictionary once + two 4-byte pointers per row
def snowflaked(n):   return KEY * n + dimprov_bytes                  # one ProvinceKey per row + DimProvince ITSELF

notes = {
 15:         "Bigger. DimProvince costs more than the 4 bytes per row it saves; the dimension is too small to pay for it.",
 1000:       "Smaller. 1,000 rows of saved pointers now outweigh the fixed cost of DimProvince.",
 100000:     "Smaller. The saving is 4 bytes per row; DimProvince is now a rounding error.",
 5000000:    "Smaller. About 20 MB saved, in the units of a fact table, a rounding error.",
 50000000:   "Smaller. About 200 MB saved, still 4 bytes per row and nothing more.",
}
rows = []
for n in [15, 1000, 100000, 5000000, 50000000]:
    fr, fe, sn = flat_raw(n), flat_encoded(n), snowflaked(n)
    rows.append((n, float(fr), fe, sn, fe - sn, "yes" if fe - sn > 0 else "NO - bigger!", notes[n]))
A = pd.DataFrame(rows, columns=["Rows in DimCity", "Flat, raw (bytes)", "Flat, encoded (bytes)", "Snowflaked (bytes)",
                                "Saving (encoded - snow)", "Snow smaller?", "What this row tells you"])
assert flat_raw(15) == 422 and flat_encoded(15) == 268 and snowflaked(15) == 292       # exact at the real 15 rows
A.style.format({"Rows in DimCity": "{:,}", "Flat, raw (bytes)": "{:,.2f}", "Flat, encoded (bytes)": "{:,}",
                "Snowflaked (bytes)": "{:,}", "Saving (encoded - snow)": "{:,}"})

,Rows in DimCity,"Flat, raw (bytes)","Flat, encoded (bytes)",Snowflaked (bytes),Saving (encoded - snow),Snow smaller?,What this row tells you
0,15,422.00,268,292,-24,NO - bigger!,Bigger. DimProvince costs more than the 4 bytes per row it saves; the dimension is too small to pay for it.
1,"1,000","28,142.71","8,148","4,232","3,916",yes,"Smaller. 1,000 rows of saved pointers now outweigh the fixed cost of DimProvince."
2,"100,000","2,814,285.57","800,148","400,232","399,916",yes,Smaller. The saving is 4 bytes per row; DimProvince is now a rounding error.
3,"5,000,000","140,714,285.57","40,000,148","20,000,232","19,999,916",yes,"Smaller. About 20 MB saved, in the units of a fact table, a rounding error."
4,"50,000,000","1,407,142,857.00","400,000,148","200,000,232","199,999,916",yes,"Smaller. About 200 MB saved, still 4 bytes per row and nothing more."


*Only non-integer values: the raw column at 1,000+ rows, because the average real city row is 394/14 bytes of text; nothing else is rounded.*

## Part B · The finding

In [3]:
saving_15  = flat_encoded(15) - snowflaked(15)
per_row    = 2 * KEY - KEY
breakeven  = (dimprov_bytes - CHAR * dict_chars) // per_row            # saving = per_row*n - (DimProvince - dictionary)
print("At 15 rows the snowflake is smaller?", saving_15 > 0, "| difference:", saving_15, "bytes")
print("Saving per city row once encoded    :", per_row, "bytes (two pointers become one)")
print("Fixed cost of DimProvince over the dictionary:", dimprov_bytes - CHAR * dict_chars, "bytes -> break-even at", breakeven, "rows")
assert saving_15 < 0 and per_row == 4
assert 4 * breakeven - (dimprov_bytes - CHAR * dict_chars) == 0
# Sensitivity: add the 'Unknown' strings to the flat dictionary too (2 columns x 7 chars)
alt = lambda n: flat_encoded(n) + CHAR * 2 * len("Unknown") - snowflaked(n)
print("If the Unknown strings are also put in the flat dictionary: saving at 15 rows =", alt(15), "bytes (break-even at", (dimprov_bytes - CHAR * (dict_chars + 14)) // per_row, "rows)")

At 15 rows the snowflake is smaller? False | difference: -24 bytes
Saving per city row once encoded    : 4 bytes (two pointers become one)
Fixed cost of DimProvince over the dictionary: 84 bytes -> break-even at 21 rows
If the Unknown strings are also put in the flat dictionary: saving at 15 rows = 4 bytes (break-even at 14 rows)


**At 15 rows the snowflake is bigger, by 24 bytes** (292 against 268). Normalizing saves 4 bytes on each of 15 city rows (60 bytes), but `DimProvince` costs 232 bytes against 148 for the dictionary it replaces, an 84-byte fixed cost. **Once encoded, the saving is exactly 4 bytes per city row** (two 4-byte pointers become one ProvinceKey); the snowflake breaks even at 21 rows and is smaller from 22.

*Sensitivity:* if the "Unknown" strings are also counted in the flat dictionary, the 15-row difference shrinks to a 4-byte saving. Either way, at 15 rows the snowflake saves nothing meaningful.

## Part C · Put it in context

In [4]:
FACT_MULT, FACT_ROW = 100, 40
C = []
for n in [15, 50_000_000]:
    sav = flat_encoded(n) - snowflaked(n); fact = n * FACT_MULT * FACT_ROW
    C.append((n, sav, fact, 100 * Fr(sav, fact)))
C = pd.DataFrame(C, columns=["Rows in DimCity", "Saving (bytes)", "Fact table bytes", "Saving as % of fact"])
C["Saving as % of fact"] = C["Saving as % of fact"].astype(float)
C["Check (|%| < 1)"] = C["Saving as % of fact"].abs().lt(1).map({True: "pass", False: "fix"})
assert (C["Check (|%| < 1)"] == "pass").all()

# even against the raw 1990s row store, the saving is still below 1% of the fact
raw_sav = flat_raw(50_000_000) - snowflaked(50_000_000)
print(f"Flat RAW vs snowflake at 50,000,000 rows: saving {float(raw_sav):,.0f} bytes = {float(100 * raw_sav / (50_000_000 * 100 * 40)):.3f}% of the fact table")
C.style.format({"Rows in DimCity": "{:,}", "Saving (bytes)": "{:,}", "Fact table bytes": "{:,}", "Saving as % of fact": "{:.6f}%"})

Flat RAW vs snowflake at 50,000,000 rows: saving 1,207,142,625 bytes = 0.604% of the fact table


,Rows in DimCity,Saving (bytes),Fact table bytes,Saving as % of fact,Check (|%| < 1)
0,15,-24,"60,000",-0.040000%,pass
1,"50,000,000","199,999,916","200,000,000,000",0.100000%,pass


**Both ends are tiny:** at 15 rows the snowflake costs 24 bytes against a 60,000-byte fact table (about -0.04%); at 50,000,000 rows it saves 199,999,916 bytes against a 200,000,000,000-byte fact table (about 0.1%). Even against the uncompressed raw row store, the saving at 50 million rows is about 0.6% of the fact.

**The one condition under which storage is worth a join:** when the dimension is so large (rows comparable to the fact table) and the repeated attribute so wide (long text, not a short name) that, in an engine with no dictionary compression, removing it saves a meaningful share of the fact's bytes. For a short, low-cardinality attribute like `Province`, in a columnar engine that already stores each string once, it never does.